# Closed-Source Models — Variant Benchmark Generation

This notebook runs closed-source language models on the morphosyntactic
variant benchmark.

The variant benchmark contains controlled linguistic modifications derived
from a subset of the original reasoning questions.

For each variant, the generation pipeline stores:

- the variant identifier,
- the corresponding base-question identifier,
- variant metadata,
- model reasoning,
- final answer,
- confidence,
- the complete raw model response,
- generation errors, if any.

This notebook handles **generation only**.

Evaluation and base–variant comparison are intentionally performed in separate
notebooks.

## Important safety and reproducibility notes

Generation is disabled by default.

No API request is sent unless:

```python
RUN_GENERATION = True

## 1. Install dependencies

This notebook uses external model APIs.

A local GPU is not required because language-model inference is performed by
the model providers.

In [25]:
%pip install -q openai anthropic google-genai pandas tqdm

In [26]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import re
import sys
import time

import pandas as pd
from tqdm.auto import tqdm


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.15
pandas: 2.2.3


## 2. Locate the project

The notebook supports two common workflows:

1. running from a cloned GitHub repository,
2. running from a copy stored in Google Drive.

The public GitHub version defaults to repository mode.

In [27]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:

    try:
        from google.colab import drive

        drive.mount("/content/drive")

        PROJECT_ROOT = DRIVE_PROJECT_PATH

    except ImportError:

        raise RuntimeError(
            "Google Drive mounting is only available in Google Colab."
        )


else:

    current_path = Path.cwd().resolve()

    candidates = [
        current_path,
        *current_path.parents,
    ]

    PROJECT_ROOT = None


    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):

            PROJECT_ROOT = candidate
            break


    if PROJECT_ROOT is None:

        raise FileNotFoundError(
            "Project root could not be located. "
            "Run the notebook from inside the repository "
            "or enable Google Drive mode."
        )


print("Project root:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


## 3. Variant dataset configuration and reproducibility

This notebook supports different versions of the variant benchmark without
hard-coding the dataset size throughout the generation pipeline.

Two dataset profiles are defined:

- **original** — the snapshot used in the original closed-model variant experiment.
- **current** — an optional evolving version for future experiments.

The `original` profile performs strict structural checks.

The `current` profile does not assume a fixed number of variants.

A SHA-256 hash is recorded so that every generated output can later be linked
to the exact dataset file that produced it.

In [28]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_PROFILE = "smoke"


DATASET_PROFILES = {

    "original": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "variants"
            / "variants.csv"
        ),

        "expected_raw_rows": 257,
        "expected_usable_rows": 257,

        "description": (
            "Original closed-model variant experiment snapshot"
        ),
    },


    "current": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "variants"
            / "variants_current.csv"
        ),

        "expected_raw_rows": None,
        "expected_usable_rows": None,

        "description": (
            "Current evolving variant benchmark"
        ),
    },

    "smoke": {
    "path": (
        PROJECT_ROOT
        / "data"
        / "variants"
        / "smoke_variants_v1.csv"
    ),

    "expected_raw_rows": 15,
    "expected_usable_rows": 15,

    "description": (
        "Small smoke-test variant dataset for pipeline validation"
    ),
    },
}


if DATASET_PROFILE not in DATASET_PROFILES:

    raise ValueError(
        f"Unknown dataset profile: {DATASET_PROFILE}"
    )


DATASET_CONFIG = DATASET_PROFILES[
    DATASET_PROFILE
]

VARIANT_DATA_PATH = DATASET_CONFIG[
    "path"
]


print("Dataset profile:")
print(f"  Name:        {DATASET_PROFILE}")
print(f"  Description: {DATASET_CONFIG['description']}")
print(f"  Path:        {VARIANT_DATA_PATH}")

Dataset profile:
  Name:        smoke
  Description: Small smoke-test variant dataset for pipeline validation
  Path:        /content/drive/MyDrive/turkish-llm-reasoning/data/variants/smoke_variants_v1.csv


In [29]:
def calculate_file_sha256(filepath):
    """
    Calculate the SHA-256 hash of a file.

    The hash identifies the exact dataset snapshot used
    by an experiment.
    """

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:

        for chunk in iter(
            lambda: file.read(8192),
            b"",
        ):

            sha256.update(chunk)

    return sha256.hexdigest()

In [30]:
if not VARIANT_DATA_PATH.exists():

    raise FileNotFoundError(
        f"Variant dataset not found: {VARIANT_DATA_PATH}"
    )


raw_variants_df = pd.read_csv(
    VARIANT_DATA_PATH
)


REQUIRED_FIELDS = [
    "variant_id",
    "base_question_id",
    "category",
    "difficulty",
    "variant_type",
    "morphological_feature",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]


missing_columns = [
    column
    for column in REQUIRED_FIELDS
    if column not in raw_variants_df.columns
]


if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


variants_df = (
    raw_variants_df
    .dropna(
        subset=REQUIRED_FIELDS
    )
    .copy()
)


DATASET_SHA256 = calculate_file_sha256(
    VARIANT_DATA_PATH
)


print(
    f"Raw variant dataset:    "
    f"{len(raw_variants_df)} rows"
)

print(
    f"Usable variant dataset: "
    f"{len(variants_df)} rows"
)

print(
    f"Excluded:               "
    f"{len(raw_variants_df) - len(variants_df)} rows"
)

print(
    f"Unique base questions:  "
    f"{variants_df['base_question_id'].nunique()}"
)

print(
    f"Dataset SHA-256:         "
    f"{DATASET_SHA256}"
)

Raw variant dataset:    15 rows
Usable variant dataset: 15 rows
Excluded:               0 rows
Unique base questions:  5
Dataset SHA-256:         08f38dc18431f08a3574af59042aefaca717015c39133a254b21a551934a2443


In [31]:
expected_raw_rows = DATASET_CONFIG[
    "expected_raw_rows"
]

expected_usable_rows = DATASET_CONFIG[
    "expected_usable_rows"
]


if (
    expected_raw_rows is not None
    and len(raw_variants_df) != expected_raw_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_raw_rows} raw rows, "
        f"but {len(raw_variants_df)} were found."
    )


if (
    expected_usable_rows is not None
    and len(variants_df) != expected_usable_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_usable_rows} usable rows, "
        f"but {len(variants_df)} were found."
    )


if not raw_variants_df["variant_id"].is_unique:

    raise ValueError(
        "variant_id values must be unique."
    )


print("✓ Variant dataset validation passed.")

✓ Variant dataset validation passed.


### Historical dataset note

The original closed-model variant experiment used a snapshot containing
**257 variants** derived from **85 base questions**.

These numbers describe the historical dataset snapshot only.

Future versions of the variant benchmark may contain additional base questions,
variant types, or individual variants.

## 4. Model configuration

The model registry separates human-readable model labels from provider-specific
API model identifiers.

New models can be added to `MODEL_REGISTRY`.

Models from an already supported provider can usually be added without changing
the rest of the notebook.

The request settings stored for historical models reflect the original
generation implementation.

A value of `None` means that the setting was not explicitly supplied to the
provider in the historical implementation.

In [32]:
# ============================================================
# GENERATION CONFIGURATION
# ============================================================

RUN_GENERATION = True

# Additional safeguard for expensive multi-model runs.
CONFIRM_MULTIPLE_MODELS = False

RUN_LABEL = "smoke_v1"

CHECKPOINT_EVERY = 5
REQUEST_DELAY_SECONDS = 0.5


# ============================================================
# MODEL REGISTRY
# ============================================================

MODEL_REGISTRY = {

    "gpt-4o-mini": {
        "provider": "openai",
        "model_id": "gpt-4o-mini",
        "temperature": 0,
        "max_tokens": 2048,
        "historical": True,
    },


    "gpt-4o": {
        "provider": "openai",
        "model_id": "gpt-4o",
        "temperature": 0,
        "max_tokens": 2048,
        "historical": True,
    },


    "claude-sonnet-4": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-20250514",

        # Not explicitly passed in the historical code.
        "temperature": None,

        "max_tokens": 2048,
        "historical": True,
    },


    "gemini-2.5-flash": {
        "provider": "google",
        "model_id": "gemini-2.5-flash",

        # Not explicitly passed in the historical code.
        "temperature": None,
        "max_tokens": None,

        "historical": True,
    },
}

In [33]:
# ============================================================
# MODEL SELECTION
# ============================================================

ORIGINAL_VARIANT_MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "claude-sonnet-4",
    "gemini-2.5-flash",
]


# Historical configuration:
#SELECTED_MODELS = ORIGINAL_VARIANT_MODELS.copy()


# For a smaller future run, this could instead be:
#
SELECTED_MODELS = [
     "gpt-4o-mini",
]


if not SELECTED_MODELS:

    raise ValueError(
        "At least one model must be selected."
    )


unknown_models = [
    model
    for model in SELECTED_MODELS
    if model not in MODEL_REGISTRY
]


if unknown_models:

    raise ValueError(
        f"Unknown model(s): {unknown_models}"
    )


if len(SELECTED_MODELS) != len(
    set(SELECTED_MODELS)
):

    raise ValueError(
        "SELECTED_MODELS contains duplicate entries."
    )


print("Selected models:")

for model_label in SELECTED_MODELS:

    config = MODEL_REGISTRY[
        model_label
    ]

    print(
        f"  - {model_label} "
        f"({config['provider']} / "
        f"{config['model_id']})"
    )

Selected models:
  - gpt-4o-mini (openai / gpt-4o-mini)


## 5. Prompt

The prompt below preserves the structure used in the original closed-model
generation experiments.

Each variant provides a context and a question.

The requested JSON response contains:

- `reasoning`
- `answer`
- `confidence`

The same prompt structure is used for base and variant generation so that the
linguistic variant itself remains the primary experimental change.

In [34]:
SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""

In [35]:
def build_user_prompt(
    context,
    question,
):
    return f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki bağlamı kullanarak soruyu cevapla. Yanıtını JSON formatında ver."""

In [36]:
PROMPT_REFERENCE = (
    SYSTEM_PROMPT
    + "\n---USER-PROMPT---\n"
    + build_user_prompt(
        "{context}",
        "{question}",
    )
)


PROMPT_SHA256 = hashlib.sha256(
    PROMPT_REFERENCE.encode("utf-8")
).hexdigest()


print("Prompt SHA-256:")
print(PROMPT_SHA256)

Prompt SHA-256:
1663da14c062cef14f02ba36cab60ef772b1e948f1ba9b8144be7f35d859e25a


## 6. API credentials

API keys must never be written directly into this notebook or committed to
GitHub.

When generation is enabled, credentials can be read from:

1. environment variables,
2. Google Colab Secrets.

Expected secret names:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GOOGLE_API_KEY`

No API request is made in this section.

In [37]:
def get_secret(name):
    """
    Read a secret from an environment variable or,
    when available, Google Colab Secrets.
    """

    value = os.getenv(name)

    if value:

        return value


    try:

        from google.colab import userdata

        return userdata.get(name)


    except Exception:

        return None

## 7. Output structure

Each selected model receives separate output files.

Results are stored under:

```text
results/raw/variants/

In [38]:
RAW_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw"
    / "variants"
)


CHECKPOINT_DIR = (
    RAW_OUTPUT_DIR
    / "checkpoints"
)


def safe_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


def get_run_name(model_label):

    return (
        f"{safe_name(DATASET_PROFILE)}__"
        f"{safe_name(model_label)}__"
        f"{safe_name(RUN_LABEL)}"
    )


def get_run_paths(model_label):

    run_name = get_run_name(
        model_label
    )

    return {
        "run_name": run_name,

        "checkpoint": (
            CHECKPOINT_DIR
            / f"{run_name}.json"
        ),

        "output": (
            RAW_OUTPUT_DIR
            / f"{run_name}.json"
        ),

        "metadata": (
            RAW_OUTPUT_DIR
            / f"{run_name}.metadata.json"
        ),
    }

In [39]:
print("Output plan:")


for model_label in SELECTED_MODELS:

    paths = get_run_paths(
        model_label
    )

    print(
        f"\n{model_label}"
    )

    print(
        f"  Checkpoint: "
        f"{paths['checkpoint']}"
    )

    print(
        f"  Output:     "
        f"{paths['output']}"
    )

    print(
        f"  Metadata:   "
        f"{paths['metadata']}"
    )

Output plan:

gpt-4o-mini
  Checkpoint: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/checkpoints/smoke__gpt-4o-mini__smoke_v1.json
  Output:     /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__gpt-4o-mini__smoke_v1.json
  Metadata:   /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__gpt-4o-mini__smoke_v1.metadata.json


## 8. JSON response parser

Models are instructed to return a JSON object.

The parser accepts:

1. direct JSON,
2. JSON inside a Markdown code block,
3. a JSON object embedded inside surrounding text.

If parsing fails, the full raw response remains stored so that the record can
later be inspected or recovered without repeating the model request.

In [40]:
def extract_json_from_response(content):
    """
    Extract a JSON object from a model response.

    Returns:
        dict or None
    """

    if content is None:

        return None


    content = str(
        content
    ).strip()


    if not content:

        return None


    # --------------------------------------------------------
    # 1. Direct JSON
    # --------------------------------------------------------

    try:

        parsed = json.loads(
            content
        )

        if isinstance(
            parsed,
            dict,
        ):

            return parsed


    except json.JSONDecodeError:

        pass


    # --------------------------------------------------------
    # 2. Markdown code block
    # --------------------------------------------------------

    code_blocks = re.findall(
        r"```(?:json)?\s*(.*?)```",
        content,
        flags=(
            re.DOTALL
            | re.IGNORECASE
        ),
    )


    for block in code_blocks:

        try:

            parsed = json.loads(
                block.strip()
            )

            if isinstance(
                parsed,
                dict,
            ):

                return parsed


        except json.JSONDecodeError:

            continue


    # --------------------------------------------------------
    # 3. JSON embedded in surrounding text
    # --------------------------------------------------------

    start = content.find("{")
    end = content.rfind("}")


    if (
        start != -1
        and end > start
    ):

        candidate = content[
            start:end + 1
        ]


        try:

            parsed = json.loads(
                candidate
            )

            if isinstance(
                parsed,
                dict,
            ):

                return parsed


        except json.JSONDecodeError:

            pass


    return None

## 9. Provider clients

Only the provider required by a model is initialized.

Adding another model from an already supported provider normally requires only
adding an entry to `MODEL_REGISTRY`.

A completely new provider requires an additional provider adapter.

In [41]:
def create_provider_client(provider):
    """
    Create a client for one supported model provider.
    """

    if provider == "openai":

        api_key = get_secret(
            "OPENAI_API_KEY"
        )

        if not api_key:

            raise RuntimeError(
                "OPENAI_API_KEY was not found."
            )

        from openai import OpenAI

        return OpenAI(
            api_key=api_key
        )


    if provider == "anthropic":

        api_key = get_secret(
            "ANTHROPIC_API_KEY"
        )

        if not api_key:

            raise RuntimeError(
                "ANTHROPIC_API_KEY was not found."
            )

        from anthropic import Anthropic

        return Anthropic(
            api_key=api_key
        )


    if provider == "google":

        api_key = get_secret(
            "GOOGLE_API_KEY"
        )

        if not api_key:

            raise RuntimeError(
                "GOOGLE_API_KEY was not found."
            )

        from google import genai

        return genai.Client(
            api_key=api_key
        )


    raise NotImplementedError(
        f"Provider not supported: {provider}"
    )

## 10. Single-response generation

Provider-specific API requests are isolated in one function.

Regardless of provider, the function returns the same internal response
structure.

The full raw response is preserved for both successful and failed parsing.

In [42]:
def generate_single_response(
    client,
    model_label,
    context,
    question,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]

    provider = model_config[
        "provider"
    ]

    model_id = model_config[
        "model_id"
    ]

    temperature = model_config[
        "temperature"
    ]

    max_tokens = model_config[
        "max_tokens"
    ]


    user_prompt = build_user_prompt(
        context,
        question,
    )


    raw_response = None


    try:

        # ====================================================
        # OpenAI
        # ====================================================

        if provider == "openai":

            request_kwargs = {
                "model": model_id,

                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },

                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],

                "response_format": {
                    "type": "json_object"
                },
            }


            if temperature is not None:

                request_kwargs[
                    "temperature"
                ] = temperature


            if max_tokens is not None:

                request_kwargs[
                    "max_tokens"
                ] = max_tokens


            response = (
                client
                .chat
                .completions
                .create(
                    **request_kwargs
                )
            )


            raw_response = (
                response
                .choices[0]
                .message
                .content
            )


        # ====================================================
        # Anthropic
        # ====================================================

        elif provider == "anthropic":

            request_kwargs = {
                "model": model_id,

                "messages": [
                    {
                        "role": "user",

                        "content": (
                            SYSTEM_PROMPT
                            + "\n\n"
                            + user_prompt
                        ),
                    }
                ],
            }


            if max_tokens is not None:

                request_kwargs[
                    "max_tokens"
                ] = max_tokens


            if temperature is not None:

                request_kwargs[
                    "temperature"
                ] = temperature


            response = (
                client
                .messages
                .create(
                    **request_kwargs
                )
            )


            text_parts = [
                block.text

                for block
                in response.content

                if getattr(
                    block,
                    "type",
                    None,
                ) == "text"
            ]


            raw_response = "\n".join(
                text_parts
            )


        # ====================================================
        # Google Gemini
        # ====================================================

        elif provider == "google":

            full_prompt = (
                SYSTEM_PROMPT
                + "\n\n"
                + user_prompt
            )


            generation_settings = {}


            if temperature is not None:

                generation_settings[
                    "temperature"
                ] = temperature


            if max_tokens is not None:

                generation_settings[
                    "max_output_tokens"
                ] = max_tokens


            if generation_settings:

                from google.genai import types

                response = (
                    client
                    .models
                    .generate_content(
                        model=model_id,
                        contents=full_prompt,

                        config=(
                            types.GenerateContentConfig(
                                **generation_settings
                            )
                        ),
                    )
                )


            else:

                response = (
                    client
                    .models
                    .generate_content(
                        model=model_id,
                        contents=full_prompt,
                    )
                )


            raw_response = (
                response.text
            )


        else:

            raise NotImplementedError(
                f"Provider not supported: "
                f"{provider}"
            )


    except Exception as exc:

        return {
            "reasoning": None,
            "answer": None,
            "confidence": None,
            "raw_response": raw_response,

            "error": (
                f"{type(exc).__name__}: "
                f"{exc}"
            ),
        }


    parsed = extract_json_from_response(
        raw_response
    )


    if parsed is None:

        return {
            "reasoning": None,
            "answer": None,
            "confidence": None,
            "raw_response": raw_response,
            "error": "JSON parsing failed",
        }


    return {
        "reasoning": parsed.get(
            "reasoning"
        ),

        "answer": parsed.get(
            "answer"
        ),

        "confidence": parsed.get(
            "confidence"
        ),

        "raw_response": raw_response,
        "error": None,
    }

In [43]:
# ============================================================
# ONE-VARIANT CANARY TEST
# ============================================================

canary_row = variants_df.iloc[0]

model_label = "gpt-4o-mini"
provider = MODEL_REGISTRY[model_label]["provider"]

print("Canary variant ID:", canary_row["variant_id"])
print("Base question ID:", canary_row["base_question_id"])

client = create_provider_client(
    provider
)

canary_result = generate_single_response(
    client=client,
    model_label=model_label,
    context=canary_row["context"],
    question=canary_row["question"],
)

print("\nError:")
print(canary_result["error"])

print("\nParsed answer:")
print(canary_result["answer"])

print("\nConfidence:")
print(canary_result["confidence"])

print("\nReasoning available:")
print(canary_result["reasoning"] is not None)

print("\nRaw response available:")
print(canary_result["raw_response"] is not None)

Canary variant ID: TR-1-018-v1
Base question ID: TR-1-018

Error:
None

Parsed answer:
10:00

Confidence:
100

Reasoning available:
True

Raw response available:
True


## 11. Run metadata

Every model run stores metadata identifying:

- the dataset,
- the dataset hash,
- the prompt hash,
- the model,
- the provider,
- request settings,
- record counts.

This allows results to be traced back to the exact experimental configuration.

In [44]:
def utc_now():

    return (
        datetime
        .now(timezone.utc)
        .isoformat()
    )


def build_run_metadata(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]


    return {
        "run_name": (
            get_run_name(
                model_label
            )
        ),

        "experiment_type": (
            "closed_models_variants"
        ),

        "dataset_profile": (
            DATASET_PROFILE
        ),

        "dataset_path": str(
            VARIANT_DATA_PATH
        ),

        "dataset_sha256": (
            DATASET_SHA256
        ),

        "raw_dataset_rows": (
            len(raw_variants_df)
        ),

        "usable_dataset_rows": (
            len(variants_df)
        ),

        "unique_base_questions": (
            int(
                variants_df[
                    "base_question_id"
                ].nunique()
            )
        ),

        "prompt_sha256": (
            PROMPT_SHA256
        ),

        "model_label": (
            model_label
        ),

        "model_id": (
            model_config[
                "model_id"
            ]
        ),

        "provider": (
            model_config[
                "provider"
            ]
        ),

        "historical_model": (
            model_config[
                "historical"
            ]
        ),

        "temperature": (
            model_config[
                "temperature"
            ]
        ),

        "max_tokens": (
            model_config[
                "max_tokens"
            ]
        ),
    }

## 12. Checkpoint helpers

Checkpoint files allow long experiments to continue after an interruption.

Before resuming, the notebook verifies that the checkpoint belongs to the same:

- dataset,
- prompt,
- model,
- provider.

This prevents results from different experiments from being mixed together.

In [45]:
def save_json_atomic(
    data,
    filepath,
):

    filepath = Path(
        filepath
    )


    filepath.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_file = (
        filepath
        .with_suffix(
            filepath.suffix + ".tmp"
        )
    )


    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


    os.replace(
        temporary_file,
        filepath,
    )

In [46]:
def load_checkpoint(
    checkpoint_file,
    current_metadata,
):

    checkpoint_file = Path(
        checkpoint_file
    )


    if not checkpoint_file.exists():

        return [], None


    with open(
        checkpoint_file,
        "r",
        encoding="utf-8",
    ) as file:

        checkpoint = json.load(
            file
        )


    saved_metadata = checkpoint[
        "metadata"
    ]

    records = checkpoint[
        "records"
    ]


    fields_to_validate = [
        "dataset_sha256",
        "prompt_sha256",
        "model_id",
        "provider",
        "temperature",
        "max_tokens"
    ]


    for field in fields_to_validate:

        if (
            saved_metadata.get(field)
            != current_metadata.get(field)
        ):

            raise ValueError(
                "Checkpoint configuration mismatch "
                f"for '{field}'."
            )


    print(
        "✓ Existing checkpoint loaded: "
        f"{len(records)} records"
    )


    return (
        records,
        saved_metadata.get(
            "created_at"
        ),
    )

## 13. Generate one model over the variant benchmark

Each model is processed independently.

A failed model run does not overwrite another model's results.

Errors remain stored as records rather than being silently removed.

Automatic retries are intentionally not performed because retries may create
additional API cost.

In [47]:
def run_model_generation(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]

    provider = model_config[
        "provider"
    ]

    paths = get_run_paths(
        model_label
    )

    metadata = build_run_metadata(
        model_label
    )


    client = create_provider_client(
        provider
    )


    records, created_at = load_checkpoint(
        paths["checkpoint"],
        metadata,
    )


    if created_at is None:

        created_at = utc_now()


    processed_ids = {
        str(
            record["variant_id"]
        )

        for record in records
    }


    remaining_df = variants_df[
        ~variants_df[
            "variant_id"
        ]
        .astype(str)
        .isin(
            processed_ids
        )
    ]


    print(
        f"\n{'=' * 60}"
    )

    print(
        f"MODEL: {model_label}"
    )

    print(
        f"{'=' * 60}"
    )

    print(
        f"Already processed: "
        f"{len(processed_ids)}"
    )

    print(
        f"Remaining: "
        f"{len(remaining_df)}"
    )


    for _, row in tqdm(
        remaining_df.iterrows(),
        total=len(remaining_df),
        desc=model_label,
    ):

        response = generate_single_response(
            client=client,
            model_label=model_label,
            context=row["context"],
            question=row["question"],
        )


        record = {
            "variant_id": str(
                row["variant_id"]
            ),

            "base_question_id": str(
                row["base_question_id"]
            ),

            "category": (
                row["category"]
            ),

            "difficulty": (
                row["difficulty"]
            ),

            "variant_type": (
                row["variant_type"]
            ),

            "morphological_feature": (
                row[
                    "morphological_feature"
                ]
            ),

            "model_label": (
                model_label
            ),

            "model_id": (
                model_config[
                    "model_id"
                ]
            ),

            "provider": (
                provider
            ),

            "model_reasoning": (
                response[
                    "reasoning"
                ]
            ),

            "model_answer": (
                response[
                    "answer"
                ]
            ),

            "model_confidence": (
                response[
                    "confidence"
                ]
            ),

            "raw_response": (
                response[
                    "raw_response"
                ]
            ),

            "error": (
                response[
                    "error"
                ]
            ),
        }


        records.append(
            record
        )


        if (
            len(records)
            % CHECKPOINT_EVERY
            == 0
        ):

            checkpoint_payload = {
                "metadata": {
                    **metadata,

                    "created_at": (
                        created_at
                    ),

                    "updated_at": (
                        utc_now()
                    ),

                    "status": (
                        "in_progress"
                    ),
                },

                "records": (
                    records
                ),
            }


            save_json_atomic(
                checkpoint_payload,
                paths["checkpoint"],
            )


        if REQUEST_DELAY_SECONDS > 0:

            time.sleep(
                REQUEST_DELAY_SECONDS
            )


    final_metadata = {
        **metadata,

        "created_at": (
            created_at
        ),

        "completed_at": (
            utc_now()
        ),

        "status": (
            "completed"
        ),

        "record_count": (
            len(records)
        ),

        "successful_records": (
            sum(
                record.get(
                    "error"
                ) is None

                for record
                in records
            )
        ),

        "error_records": (
            sum(
                record.get(
                    "error"
                ) is not None

                for record
                in records
            )
        ),
    }


    final_payload = {
        "metadata": (
            final_metadata
        ),

        "records": (
            records
        ),
    }


    save_json_atomic(
        final_payload,
        paths["output"],
    )


    save_json_atomic(
        final_metadata,
        paths["metadata"],
    )


    save_json_atomic(
        final_payload,
        paths["checkpoint"],
    )


    print(
        f"\n✓ {model_label} completed."
    )

    print(
        f"Records: "
        f"{len(records)}"
    )

    print(
        f"Output: "
        f"{paths['output']}"
    )


    return pd.DataFrame(
        records
    )

## 14. Generation safety gate

The next cell is the only section that starts generation.

While `RUN_GENERATION = False`, no model request is sent.

If multiple models are selected, `CONFIRM_MULTIPLE_MODELS` must also be set to
`True`.

This additional confirmation reduces the risk of accidentally starting a large
and potentially expensive batch of API requests.

In [48]:
generation_results = {}


if RUN_GENERATION:

    if (
        len(SELECTED_MODELS) > 1
        and not CONFIRM_MULTIPLE_MODELS
    ):

        raise RuntimeError(
            "Multiple models are selected. "
            "Set CONFIRM_MULTIPLE_MODELS = True "
            "only if you intentionally want to run "
            "the full multi-model batch."
        )


    print(
        "Generation is ENABLED."
    )

    print(
        f"Variants per model: "
        f"{len(variants_df)}"
    )

    print(
        f"Selected models: "
        f"{len(SELECTED_MODELS)}"
    )

    print(
        f"Maximum planned requests: "
        f"{len(variants_df) * len(SELECTED_MODELS)}"
    )


    for model_label in SELECTED_MODELS:

        generation_results[
            model_label
        ] = run_model_generation(
            model_label
        )


else:

    print(
        "Generation is disabled."
    )

    print(
        "No API request was sent."
    )

    print(
        "Set RUN_GENERATION = True "
        "only when intentionally running "
        "the experiment."
    )

Generation is ENABLED.
Variants per model: 15
Selected models: 1
Maximum planned requests: 15

MODEL: gpt-4o-mini
Already processed: 0
Remaining: 15


gpt-4o-mini:   0%|          | 0/15 [00:00<?, ?it/s]


✓ gpt-4o-mini completed.
Records: 15
Output: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/variants/smoke__gpt-4o-mini__smoke_v1.json


## 15. End of variant generation notebook

This notebook intentionally ends after raw variant generation.

It does not calculate:

- answer accuracy,
- faithfulness,
- calibration,
- hallucination-risk metrics,
- base–variant differences,
- statistical comparisons,
- figures.

Those stages are separated from generation so that the saved model responses
can be re-evaluated without repeating potentially expensive API calls.

The next generation notebooks cover open-source models:

- `03_open_models_base.ipynb`
- `04_open_models_variants.ipynb`

Saved outputs from all generation notebooks will later be evaluated in:

`05_evaluate_saved_outputs.ipynb`